# Laboratorium 6. Czy do przewidywania salda potrzebujemy i Limit, i Rating?

Zapisz kopię notebooka na swoim Dysku. Wpisuj kod w pustych komórkach. W odpowiedziach odwołuj się do otrzymanych liczb lub wykresów.

## Przygotowanie

W danych *Credit* jeden wiersz opisuje klienta. Przewidujemy średnie saldo zadłużenia na karcie (`Balance`, USD). `Limit` to limit kredytowy w USD, `Rating` to ocena kredytowa w punktach, `Income` to dochód w tysiącach USD, a `Student` przyjmuje wartość 1 dla studenta. Uruchom komórkę. W częściach 1--4 korzystaj tylko z próby uczącej (`X_tr`, `y_tr`, 300 klientów). Do próby testowej sięgnij w części 5.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_path = Path("../dane/Credit.csv")
data_url = "https://rodakt.github.io/MAD/laby/dane/Credit.csv"
credit = pd.read_csv(data_path if data_path.is_file() else data_url)
features = ["Limit", "Rating", "Income", "Student"]
X = credit[features].copy()
X["Student"] = X["Student"].map({"No": 0, "Yes": 1})
X_tr, X_te, y_tr, y_te = train_test_split(
    X, credit["Balance"], test_size=100, random_state=707
)
print(f"Próba ucząca: {len(X_tr)} klientów. Próba testowa: {len(X_te)} klientów.")

## 1. Co wnosi druga cecha?

Chcemy sprawdzić, ile do przewidywania salda (`Balance`) wnosi ocena kredytowa (`Rating`), gdy znamy już limit (`Limit`). Najpierw sprawdź, jak silnie te dwie cechy są ze sobą związane. Narysuj ocenę kredytową względem limitu i oblicz ich korelację.

Jako punkt odniesienia dopasuj model OLS przewidujący saldo na podstawie samego limitu. O ile dolarów różnią się przewidywane salda dwóch klientów, których limity różnią się o 1000 USD?

<details><summary>Podpowiedź</summary>

`LinearRegression().fit(X_tr[["Limit"]], y_tr)` dopasowuje model z wyrazem wolnym. Nachylenia są w atrybucie `coef_`.

</details>

**Twoja odpowiedź**

### Co zmieni dodanie oceny kredytowej?

Wróć do wykresu oceny kredytowej względem limitu. Czy wśród klientów o podobnej ocenie widzisz duże różnice limitu? Dlaczego taki układ punktów może utrudniać rozdzielenie związku limitu z saldem od związku oceny kredytowej z saldem?

Dopasuj model OLS z limitem i oceną kredytową. Pomnóż współczynnik przy limicie przez 1000 i porównaj wynik z otrzymanym w pierwszym modelu. Następnie dla każdego klienta z próby uczącej oblicz różnicę między saldami przewidywanymi przez oba modele. Policz pierwiastek ze średniej kwadratów tych różnic i porównaj go z odchyleniem standardowym salda w próbie uczącej.

O ile zmieniła się przewidywana różnica sald przypadająca na 1000 USD różnicy w limicie? Czy zmianie współczynnika towarzyszyła duża zmiana przewidywanych sald na tle ich zróżnicowania w próbie? Opisz, jakich dwóch klientów porównujesz, gdy przeliczasz ten współczynnik na 1000 USD różnicy w limicie.

<details><summary>Podpowiedź</summary>

Oba wektory predykcji powinny odpowiadać tym samym wierszom w tej samej kolejności.

</details>

**Twoja odpowiedź**

## 2. Czy jednostka limitu wpływa na predykcję ridge?

Przewidujesz saldo na podstawie limitu i oceny kredytowej. Przygotuj dwie wersje tych samych danych uczących. W pierwszej zostaw limit w dolarach, w drugiej podziel go przez 1000. Rozważ klienta z limitem 4000 USD. Po zmianie jednostki jego limit zapisujesz jako 4. Jak trzeba zmienić współczynnik przy limicie, żeby jego iloczyn z limitem pozostał taki sam? Jak wpłynie to na kwadrat współczynnika w karze ridge?

W każdej wersji danych dopasuj model ridge z `alpha=100`. Dla każdego klienta oblicz różnicę między predykcjami, a następnie pierwiastek ze średniej kwadratów tych różnic (w USD). Powtórz całe porównanie, tym razem standaryzując obie cechy przed dopasowaniem każdego modelu. Jak zmienił się wynik porównania? Aby wyjaśnić wynik po standaryzacji, porównaj standaryzowane wartości limitu tych samych klientów w obu wersjach danych.

<details><summary>Podpowiedź</summary>

`make_pipeline(StandardScaler(), Ridge(alpha=100))` tworzy potok. Dopasowujesz go i przewidujesz tak samo jak pojedynczy model.

</details>

**Twoja odpowiedź**

## 3. Które współczynniki reagują na karę?

Wróć do limitu w dolarach. Do limitu i oceny kredytowej dodaj dochód i status studenta. Dopasuj model ridge z `alpha=1` po standaryzacji wszystkich czterech cech. Odczytaj współczynniki i przypisz je do nazw kolumn. Jakiej zmianie limitu w oryginalnych jednostkach odpowiada wzrost standaryzowanego limitu o 1? Co współczynnik przy limicie mówi wtedy o różnicy przewidywanych sald przy tych samych wartościach pozostałych cech?

<details><summary>Podpowiedź</summary>

Jeśli dopasowany potok ma nazwę `model`, to `model[-1].coef_` zwraca współczynniki w kolejności kolumn użytych przy dopasowaniu. W tej samej kolejności `model[0].scale_` podaje odchylenia standardowe, których użyto do standaryzacji.

</details>

**Twoja odpowiedź**

### Od jednego dopasowania do dwóch ścieżek

Zbadaj, jak wraz ze wzrostem kary zmieniają się współczynniki przy tych samych czterech standaryzowanych cechach. Dla każdej metody przygotuj siatkę 31 wartości `alpha` rozmieszczonych geometrycznie. Dla ridge przyjmij zakres od 0,001 do 100 000, a dla LASSO od 0,001 do 1000. W pętli po wartościach kary dopasuj model i zapisz jego cztery współczynniki.

Dla każdej metody narysuj osobny wykres. Na osi poziomej zaznacz `alpha` w skali logarytmicznej, a każdą z czterech linii podpisz nazwą cechy. Porównaj współczynnik przy limicie dla najsłabszej i najsilniejszej badanej kary ridge. Czy na całej ścieżce zmienia się w tym samym kierunku? W razie wątpliwości zajrzyj do zapisanych wartości. W zapisanych wynikach LASSO znajdź najmniejszą badaną karę, przy której współczynnik przy `Limit` lub `Rating` wynosi zero. Jaki współczynnik ma wtedy druga cecha z tej pary? Czy na podstawie samego wykresu z części 1 można było rozstrzygnąć, którą z tych dwóch cech zachowa LASSO? Jakiej zmiennej na tym wykresie brakuje?

<details><summary>Podpowiedź</summary>

`np.geomspace(start, stop, 31)` tworzy siatkę kar. Aby dokładniej wyznaczyć współczynniki LASSO, ustaw `max_iter=100000` i `tol=1e-8`. Parametr `alpha` ma w obu metodach różną skalę.

</details>

**Twoja odpowiedź**

## 4. Którą procedurę wybierzesz?

Sprawdź, ile dla trafności przewidywań wnosi każda z dwóch powiązanych cech. Porównaj trzy modele OLS przewidujące saldo. Pierwszy wykorzystuje wszystkie cztery cechy. W drugim pomiń tylko limit, a w trzecim tylko ocenę kredytową.

Podziel próbę uczącą na pięć części za pomocą `KFold(n_splits=5, shuffle=True, random_state=708)`. Oceń każdy model na tych samych podziałach i oblicz jego średni błąd średniokwadratowy (MSE) z pięciu części walidacyjnych. O ile zmienia się średni MSE względem pełnego modelu po pominięciu limitu, a o ile po pominięciu oceny?

<details><summary>Podpowiedź</summary>

Dla listy wybranych kolumn `features_used` wywołanie `cross_val_score(model, X_tr[features_used], y_tr, cv=cv, scoring="neg_mean_squared_error")` zwraca pięć wyników ze znakiem minus. Zmień ich znak przed obliczeniem średniego MSE.

</details>

**Twoja odpowiedź**

### Dobierz karę i zapisz decyzję

Teraz porównaj trzy modele OLS z ridge i LASSO. Dla obu metod regularyzacji użyj wszystkich czterech cech. Dla każdej kary z odpowiedniej siatki z części 3 oceń potok ze standaryzacją na tych samych pięciu podziałach co OLS. Z których obserwacji potok wyznacza średnie i odchylenia standardowe cech w danym podziale? Dlaczego ma to znaczenie przy ocenie błędu?

Dla każdej metody regularyzacji narysuj średni walidacyjny MSE w funkcji kary. Odczytaj karę dającą najmniejszy średni MSE i zestaw oba minima z wynikami trzech modeli OLS. Spośród tych pięciu wyników wybierz najmniejszy. Zapisz metodę, zestaw cech i, jeśli model ma karę, wartość `alpha`. O ile mniejszy jest średni MSE wybranego modelu od drugiego wyniku w tym zestawieniu?

<details><summary>Podpowiedź</summary>

Dla każdej wartości kary zachowaj pięć walidacyjnych MSE. Ich średniej użyjesz do narysowania krzywej, a poszczególnych wartości do obliczeń w zadaniu opcjonalnym.

</details>

**Twoja odpowiedź**

## 5. Jak dobrze przewidujesz saldo nowego klienta?

Dopasuj model wybrany w części 4 na całej próbie uczącej (300 klientów). Użyj go do przewidzenia salda każdego ze 100 klientów z próby testowej. Porównaj predykcje z rzeczywistymi saldami (`y_te`) i oblicz testowy RMSE.

Zapisz dwa wnioski. Jak duży jest błąd przewidywania salda nowego klienta, jeśli mierzymy go przez RMSE w dolarach? Wróć do porównania trzech modeli OLS z części 4. Który zestaw cech wybrałbyś spośród tych trzech, gdyby kryterium był najmniejszy średni MSE? Podaj różnicę błędu względem każdego z pozostałych zestawów i na tej podstawie odpowiedz na pytanie o przydatność obu cech w tych danych.

<details><summary>Podpowiedź</summary>

Użyj tych samych kolumn i ustawień co przy wyborze. Pierwiastek z MSE przywraca jednostkę salda.

</details>

**Twoja odpowiedź**

## Opcjonalnie. Jak silną karę dopuszcza reguła 1 SE?

Wróć do pięciu walidacyjnych MSE zapisanych dla każdej kary LASSO w części 4. Znajdź karę o najmniejszym średnim MSE. Dla tej kary oblicz błąd standardowy średniej (SE). W tym celu podziel odchylenie standardowe pięciu MSE (z dzielnikiem 4) przez pierwiastek z 5.

Dodaj obliczony SE do najmniejszego średniego MSE. Spośród kar, dla których średni MSE nie przekracza tego progu, wybierz największą. W wynikach z części 3 wypisz nazwy cech z niezerowymi współczynnikami LASSO dla kary wybranej według reguły 1 SE i dla kary dającej minimum. Czy silniejsza kara zmieniła zestaw cech używanych przez model?

<details><summary>Podpowiedź</summary>

Ustal próg raz, na podstawie wyników przy minimum.

</details>

**Twoja odpowiedź**